# IndiVoice-DeepASR: Cloud Gateway 🚀

This notebook is your command center for training the **IndiVoice-DeepASR** model using Google Colab GPUs. 

### 📁 Google Drive Setup (Important)
To ensure your data and model checkpoints are saved permanently, we use Google Drive. Ensure you have the following structure in your Drive:
`My Drive/IndiVoice-DeepASR/data/raw/` 

1. **Upload your audio files** (.wav) to the `raw` folder on your Drive.
2. **Upload your transcripts** (.txt) to the same folder or a subfolder.

## 1. Environment & Drive Initialization

In [ ]:
# 1. Verify GPU
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 3. Configure Paths
DRIVE_BASE = '/content/drive/MyDrive/IndiVoice-DeepASR'
os.makedirs(DRIVE_BASE, exist_ok=True)

# 4. Sync Repository
%cd "{DRIVE_BASE}"
if not os.path.exists('.git'):
    !git clone https://github.com/purvanshjoshi/IndiVoice-DeepASR.git .
else:
    !git pull origin main

# 5. Install Dependencies
!pip install -r requirements.txt

print("\n✅ Setup Complete. Working directory:", os.getcwd())

In [ ]:
# 🔍 Directory Verification
raw_data_path = 'data/raw'
if os.path.exists(raw_data_path):
    files = os.listdir(raw_data_path)
    print(f"Found {len(files)} items in {raw_data_path}")
else:
    print(f"❌ ERROR: {raw_data_path} not found. Please create it in your Google Drive.")

## 2. Phase 1: Preprocessing
Standardizes audio to 16kHz Mono and generates the manifest for training.

In [ ]:
!python src/preprocess.py \
    --input_dir data/raw \
    --output_dir data/processed \
    --transcript_dir data/raw \
    --manifest_path data/processed/train_manifest.json

## 3. Phase 2: Deep Learning Training
Fine-tunes Whisper-Medium using LoRA. Best checkpoints are saved to your Drive.

In [ ]:
!python src/train.py \
    --train_manifest data/processed/train_manifest.json \
    --val_manifest data/processed/train_manifest.json \
    --output_dir models/whisper-indian-lora

## 4. Phase 3: Evaluation
Test the performance on unseen data.

In [ ]:
!python src/evaluate.py \
    --model_path models/whisper-indian-lora/final \
    --test_manifest data/processed/train_manifest.json

## 5. Phase 4: Launch Demo
Starts a Gradio web app. Click the public `gradio.live` link to try your model.

In [ ]:
!python src/deploy.py --model_path models/whisper-indian-lora/final --share